<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.2-solid-oxide-cell/Ex10.2_04_optimisation.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_10.2 · Notebook 04 — lifetime-aware optimisation

**Paired with L10.2 · Solid oxide cells**

**The course finale.** Everything from L8 to L10 arrives here.

Optimise a 24-hour operating trajectory — current density and temperature —
against a time-varying electricity price, subject to an end-of-life constraint
on the area-specific resistance.

$$\max_{i(t),\,T(t)} \int \left[p_{H_2}\dot n_{H_2}
- c_e\,iAV\right]dt
\qquad \text{s.t.} \quad \mathrm{ASR}(t_{life}) \le \mathrm{ASR}_{max}$$

Nothing here trains a PINN. The decision variables are the trajectory itself,
and what is being differentiated through is the cell model in `problem.py` —
which is why every function in it works on tensors as well as arrays.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.2-solid-oxide-cell/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import course_core as cc
cc.keep_outputs("Ex10.2_outputs")


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

In [ ]:
par = pb.SOCParams("SOEC", 1073.15, 0.10, 0.90)
price = pb.price_profile(24, "daily")
plt.figure(figsize=(7, 2.6)); plt.step(np.arange(24), price, where="mid")
plt.xlabel("hour"); plt.ylabel("electricity price"); plt.tight_layout(); plt.show()

## TODO 1 — a baseline

Run at constant current density and evaluate. This is what a plant without
optimisation does, and it is the number you must beat.

In [ ]:
base = pb.evaluate_trajectory(np.full(24, 1.0), np.full(24, par.T), par, price)
print(f"baseline profit {base['profit']:.2f}   ASR_end {base['ASR_end']:.4f}")
pb.plot_trajectory(base, price)

## TODO 2 — optimise the trajectory

Two routes, and you should do **both**:

**(a) Gradient-based, through a differentiable model.** Represent the
trajectory in torch, evaluate the objective, and let autograd supply
∂J/∂u. This is the point of L10.2's *Why a Differentiable Model Changes the Problem*.

**(b) Brute force on the reference solver.** A coarse grid or random search.
Slow, but it owes nothing to the surrogate.

Then compare. If they agree, you have evidence the differentiable route works.
If they disagree, the optimiser has found an error in your model and exploited
it — *Where to Be Careful* in L10.2, met in person. **Either outcome is a result.**

In [ ]:
# TODO (a): gradient-based optimisation of i(t) and T(t)
#   - parametrise the trajectory as torch tensors with requires_grad=True
#   - build the objective from pb.evaluate_trajectory's physics, in torch
#   - penalise the ASR constraint, then step Adam
#
#   i_t = to_tensor(np.full(24, 1.0).reshape(-1, 1), requires_grad=True)
#   T_t = to_tensor(np.full(24, par.T).reshape(-1, 1), requires_grad=True)
#
#   Reshape to a column first: to_tensor calls np.atleast_2d, so a bare
#   (24,) array comes back as a (1, 24) row and every later broadcast is wrong.
#
#   pb.cell_voltage, pb.eta_* and pb.hydrogen_rate all dispatch on
#   torch.is_tensor, so the objective is the same expression as in
#   pb.evaluate_trajectory — but note that pb.evaluate_trajectory itself is
#   NumPy throughout (it builds a fresh SOCParams per hour), so rebuild the
#   objective rather than calling it, and use it to score the answer at the end.
#
#   train_two_stage optimises model.parameters(); these are free tensors, not
#   a module, so drive torch.optim.Adam([i_t, T_t], lr=...) directly:
#
#       opt = torch.optim.Adam([i_t, T_t], lr=1e-2)
#       for step in range(N):
#           opt.zero_grad()
#           J = -(objective(i_t, T_t)) + w * constraint_penalty(i_t, T_t)
#           J.backward(); opt.step()
#
#   Keep the loss history and plot it with plot_curves({"adam": history}).

raise NotImplementedError

In [ ]:
# TODO (b): brute-force search on the reference solver, for comparison
#
#   pb.evaluate_trajectory is the reference: give it a candidate (i, T) pair
#   and read "profit" and "ASR_end" back. A coarse grid over a few constant
#   levels, or a random search over 24-hour trajectories, is enough.
#
#   Report both answers side by side and say which you trust.

raise NotImplementedError

## TODO 3 — vary the required lifetime

Re-optimise for a required lifetime of 5 000, 20 000 and 40 000 hours. Watch
the optimal trajectory move.

This is the clearest demonstration in the whole course that temperature is a
**trade-off and not a setting**: the same cell, the same prices, and a
different answer purely because the device must last longer.

In [ ]:
# TODO: loop over required lifetimes, re-optimise, and overlay the trajectories
#
#   for t_life in (5_000, 20_000, 40_000):
#       ... re-optimise with the ASR constraint scaled to that lifetime ...
#
#   Check the constraint is actually binding. An unconstrained optimisation
#   will happily destroy the cell for profit, and a suspiciously large margin
#   over the baseline usually means it did.

raise NotImplementedError

## Save

Notebook 05 reads every case it finds in `Ex10.2_outputs` and puts one row in
the report per case.

In [ ]:
import pickle

os.makedirs(cc.OUTPUT_DIR, exist_ok=True)
# with open(os.path.join(cc.OUTPUT_DIR, "ex102_opt.pkl"), "wb") as f:
#     pickle.dump([{"par": par, "OCV": float(pb.nernst(par)),
#                   "i_tn": ..., "profit": ..., "ASR_end": ...}], f)
#
# After writing it, so that it survives a Colab session without Drive:
#     cc.saved(os.path.join(cc.OUTPUT_DIR, "ex102_opt.pkl"))


---

## 1 · Before you move on

Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which, and the Questions slide at the end of the lecture has them in full.

1. Route (a) gets $\partial J/\partial u$ from autograd, and route (b) evaluates the reference model many times. Count the decision variables in a 24-hour trajectory of $i(t)$ and $T(t)$, and say what finite differences would cost per iteration. Why does optimisation want a differentiable model rather than just an accurate one?
   *→ L10.2 Q9*
2. If the two routes disagree, the optimiser may have found an error in your model and exploited it. How would you tell that apart from a genuinely better optimum? Which check in TODO 3 catches a trajectory that destroys the cell for profit, and why does a differentiable model still need to be accurate where the optimiser takes it?
   *→ L10.2 Q9*
3. Re-optimise for required lifetimes of 5 000, 20 000 and 40 000 hours. How does the optimal temperature move as the required lifetime grows, and why? Explain it through what degradation depends on and with what shape, and say whether the ASR constraint was binding in each case.
   *→ L10.2 Q6*
4. This notebook trains no PINN; it differentiates through the cell model. Name the fields that L8, L9 and L10.1 predicted, and say which term of this objective or its constraints each would supply in a distributed version. Which constraint would need a temperature field, and why can the 0-D model here not impose it?
   *→ L10.2 Q8, Q10*


*Write your answers here. You will copy them into the report in notebook 05, which adds them to what you submit.*

1.
2.
3.
4.

---

Continue with **[`Ex10.2_05_report.ipynb`](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.2-solid-oxide-cell/Ex10.2_05_report.ipynb)**.
